In [2]:
import torch

In [3]:
#our dataset location to read
data_location = r"C:\Users\venuv\Documents\ai developer\public\tinyshakesphere.txt".replace("\\", "/")
data_location

'C:/Users/venuv/Documents/ai developer/public/tinyshakesphere.txt'

In [4]:
with open(data_location, "r") as f:
    text = f.read()
print(text[0:100])


First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [5]:
uniqueChars = sorted(list(set(text)))

In [6]:
vocabSize = len(uniqueChars)

In [7]:
#create a dictionary to map characters to integers and from integers to characters
stoi = {ch:i for i, ch in enumerate(uniqueChars)}
itos = {i:ch for i, ch in enumerate(uniqueChars)}

In [8]:
#convert data into numbers
encoded = [stoi[ch] for ch in text]


In [9]:
data = torch.tensor(encoded, dtype=torch.long)

In [10]:
print(data.shape)

torch.Size([1115394])


In [11]:
n = int(0.9*len(data))
training = data[0:n]
testing = data[n:]


In [12]:
block_size=8
batch_size=4

In [13]:
def get_batch(split):

    data = training if split == 'train' else testing

    ix = torch.randint(
        len(data) - block_size,
        (batch_size,)
    )

    x = torch.stack([
        data[i:i+block_size]
        for i in ix
    ])

    y = torch.stack([
        data[i+1:i+block_size+1]
        for i in ix
    ])

    return x,y

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [15]:
# #combine everything in a single class
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# class MiniGPT(nn.Module):

#     def __init__(
#         self,
#         vocabSize,
#         embedding_size,
#         block_size,
#         num_heads
#     ):

#         super().__init__()

#         self.embedding_size = embedding_size
#         self.block_size = block_size
#         self.num_heads = num_heads
#         self.head_size = embedding_size // num_heads

#         self.dropout = nn.Dropout(0.2)

#         # token embeddings
#         self.tokenEmbedding = nn.Embedding(
#             vocabSize,
#             embedding_size
#         )

#         # positional embeddings
#         self.positionEmbedding = nn.Embedding(
#             block_size,
#             embedding_size
#         )

#         # attention layers
#         self.query = nn.Linear(
#             embedding_size,
#             embedding_size,
#             bias=False
#         )

#         self.key = nn.Linear(
#             embedding_size,
#             embedding_size,
#             bias=False
#         )

#         self.value = nn.Linear(
#             embedding_size,
#             embedding_size,
#             bias=False
#         )

#         self.outputProjection = nn.Linear(
#             embedding_size,
#             embedding_size
#         )

#         # layer norms
#         self.layerNorm1 = nn.LayerNorm(
#             embedding_size
#         )

#         self.layerNorm2 = nn.LayerNorm(
#             embedding_size
#         )

#         # feed forward network
#         self.ffn = nn.Sequential(

#             nn.Linear(
#                 embedding_size,
#                 4 * embedding_size
#             ),

#             nn.ReLU(),

#             nn.Linear(
#                 4 * embedding_size,
#                 embedding_size
#             )
#         )

#         # final vocab projection
#         self.lm_head = nn.Linear(
#             embedding_size,
#             vocabSize
#         )

#     def forward(
#     self,
#     x,
#     targets=None,
#     k_cache=None,
#     v_cache=None
# ):

#         B,T = x.shape

#         # token embeddings
#         tokenEmbeddings = self.tokenEmbedding(x)

#         # positional embeddings
#         positions = torch.arange(
#             T,
#             device=x.device
#         )

#         positionEmbeddings = self.positionEmbedding(
#             positions
#         )

#         x = tokenEmbeddings + positionEmbeddings

#         # ==================================================
#         # TRAINING MODE
#         # ==================================================

#         if targets is not None:

#             # full sequence QKV
#             Q = self.query(x)
#             K = self.key(x)
#             V = self.value(x)

#             # split heads
#             Q = Q.view(
#                 B,
#                 T,
#                 self.num_heads,
#                 self.head_size
#             ).permute(0,2,1,3)

#             K = K.view(
#                 B,
#                 T,
#                 self.num_heads,
#                 self.head_size
#             ).permute(0,2,1,3)

#             V = V.view(
#                 B,
#                 T,
#                 self.num_heads,
#                 self.head_size
#             ).permute(0,2,1,3)

#             # attention scores
#             attention = (
#                 Q @ K.transpose(-2,-1)
#             ) / (self.head_size ** 0.5)

#             # causal mask
#             mask = torch.tril(
#                 torch.ones(
#                     T,
#                     T,
#                     device=x.device
#                 )
#             )

#             attention = attention.masked_fill(
#                 mask == 0,
#                 float('-inf')
#             )

#             # softmax
#             attention = F.softmax(
#                 attention,
#                 dim=-1
#             )

#             # weighted values
#             out = attention @ V

#             # merge heads
#             out = out.permute(
#                 0,2,1,3
#             ).contiguous().view(
#                 B,
#                 T,
#                 self.embedding_size
#             )

#             out = self.outputProjection(out)

#             out = self.dropout(out)

#             # residual + norm
#             x = self.layerNorm1(x + out)

#             # FFN
#             ffnOut = self.ffn(x)

#             ffnOut = self.dropout(ffnOut)

#             # residual + norm
#             x = self.layerNorm2(x + ffnOut)

#             # logits
#             logits = self.lm_head(x)

#             # loss
#             B,T,C = logits.shape

#             logitsFlat = logits.view(
#                 B*T,
#                 C
#             )

#             targetsFlat = targets.view(B*T)

#             loss = F.cross_entropy(
#                 logitsFlat,
#                 targetsFlat
#             )

#             return logits, loss

#         # ==================================================
#         # KV CACHE INFERENCE MODE
#         # ==================================================

#         else:

#             # only newest token
#             x = x[:, -1:, :]

#             # new QKV
#             Q = self.query(x)
#             Knew = self.key(x)
#             Vnew = self.value(x)

#             # initialize cache
#             if k_cache is None:
#                 k_cache = Knew
#                 v_cache = Vnew

#             else:

#                 k_cache = torch.cat(
#                     [k_cache, Knew],
#                     dim=1
#                 )

#                 v_cache = torch.cat(
#                     [v_cache, Vnew],
#                     dim=1
#                 )

#             Q = Q
#             K = k_cache
#             V = v_cache

#             Tq = Q.shape[1]
#             Tk = K.shape[1]

#             # split heads
#             Q = Q.view(
#                 B,
#                 Tq,
#                 self.num_heads,
#                 self.head_size
#             ).permute(0,2,1,3)

#             K = K.view(
#                 B,
#                 Tk,
#                 self.num_heads,
#                 self.head_size
#             ).permute(0,2,1,3)

#             V = V.view(
#                 B,
#                 Tk,
#                 self.num_heads,
#                 self.head_size
#             ).permute(0,2,1,3)

#             # attention
#             attention = (
#                 Q @ K.transpose(-2,-1)
#             ) / (self.head_size ** 0.5)

#             # NO MASK NEEDED

#             attention = F.softmax(
#                 attention,
#                 dim=-1
#             )

#             # weighted values
#             out = attention @ V

#             # merge heads
#             out = out.permute(
#                 0,2,1,3
#             ).contiguous().view(
#                 B,
#                 Tq,
#                 self.embedding_size
#             )

#             out = self.outputProjection(out)

#             # residual + norm
#             x = self.layerNorm1(x + out)

#             # FFN
#             ffnOut = self.ffn(x)

#             # residual + norm
#             x = self.layerNorm2(x + ffnOut)

#             # logits
#             logits = self.lm_head(x)

#             return logits, None, k_cache, v_cache
    
#     def generate(
#     self,
#     x,
#     max_new_tokens,
#     Temperature
#     ):
#         k_cache = None
#         v_cache = None

#         for _ in range(max_new_tokens):

#             # keep only recent context
#             x_cond = x[:, -self.block_size:]
#             print(x_cond)

#             logits, _, k_cache, v_cache = self(
#                 x_cond,
#                 targets=None,
#                 k_cache=k_cache,
#                 v_cache=v_cache
#             )

#             # take last token predictions
#             logits = logits[:, -1, :]

#             #use temperature to control response randomness
#             logits/=Temperature

#             # convert to probabilities
#             probabilities = F.softmax(
#                 logits,
#                 dim=-1
#             )

#             # sample next token
#             nextToken = torch.multinomial(
#                 probabilities,
#                 num_samples=1
#             )

#             # append token
#             x = torch.cat(
#                 (x, nextToken),
#                 dim=1
#             )

#         return x

In [26]:
class RmsNorm(nn.Module):
    def __init__(
            self,
            dim,
            eps=1e-06
            ):
        super().__init__()
        self.dim = dim
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = torch.sqrt(torch.mean(
            x*x,
            dim=-1,
            keepdim=True
            )+self.eps
        )

        x=x/rms

        return x*self.weight

        

In [27]:

class TransformerBlock(nn.Module):

    def __init__(
        self,
        embedding_size,
        num_heads
    ):

        super().__init__()

        self.embedding_size = embedding_size
        self.num_heads = num_heads
        self.head_size = embedding_size // num_heads

        self.query = nn.Linear(
            embedding_size,
            embedding_size,
            bias=False
        )

        self.key = nn.Linear(
            embedding_size,
            embedding_size,
            bias=False
        )

        self.value = nn.Linear(
            embedding_size,
            embedding_size,
            bias=False
        )

        self.outputProjection = nn.Linear(
            embedding_size,
            embedding_size
        )

        self.layerNorm1 = nn.LayerNorm(
            embedding_size
        )

        self.layerNorm2 = nn.LayerNorm(
            embedding_size
        )

        self.ffn = nn.Sequential(

            nn.Linear(
                embedding_size,
                4 * embedding_size
            ),

            nn.ReLU(),

            nn.Linear(
                4 * embedding_size,
                embedding_size
            )
        )

        self.norm1 = RmsNorm(embedding_size)
        self.norm2 = RmsNorm(embedding_size)

    def forward(self, x):

        B,T,C = x.shape

        xNorm = self.norm1(x)

        Q = self.query(xNorm)
        K = self.key(xNorm)
        V = self.value(xNorm)

        Q = Q.view(
            B,T,
            self.num_heads,
            self.head_size
        ).permute(0,2,1,3)

        K = K.view(
            B,T,
            self.num_heads,
            self.head_size
        ).permute(0,2,1,3)

        V = V.view(
            B,T,
            self.num_heads,
            self.head_size
        ).permute(0,2,1,3)

        attention = (
            Q @ K.transpose(-2,-1)
        ) / (self.head_size ** 0.5)

        mask = torch.tril(
            torch.ones(
                T,
                T,
                device=x.device
            )
        )

        attention = attention.masked_fill(
            mask == 0,
            float('-inf')
        )

        attention = F.softmax(
            attention,
            dim=-1
        )

        out = attention @ V

        out = out.permute(
            0,2,1,3
        ).contiguous().view(
            B,T,C
        )

        out = self.outputProjection(out)

        x = x+out

        xNorm = self.norm2(x)

        ffnOut = self.ffn(xNorm)

        x = x+ffnOut

        return x

In [28]:
#combine everything in a single class


class MiniGPT(nn.Module):

    def __init__(
        self,
        vocabSize,
        embedding_size,
        block_size,
        num_heads
    ):

        super().__init__()

        self.embedding_size = embedding_size
        self.block_size = block_size
        self.num_heads = num_heads
        self.head_size = embedding_size // num_heads

        self.num_layers = 4

        self.dropout = nn.Dropout(0.2)

        # token embeddings
        self.tokenEmbedding = nn.Embedding(
            vocabSize,
            embedding_size
        )

        # positional embeddings
        self.positionEmbedding = nn.Embedding(
            block_size,
            embedding_size
        )

        

        # final vocab projection
        self.lm_head = nn.Linear(
            embedding_size,
            vocabSize
        )
    
        self.blocks = nn.Sequential(

            *[
                TransformerBlock(
                    self.embedding_size,
                    self.num_heads
                )

                for _ in range(self.num_layers)
            ]
        )

    def forward(
        self,
        x,
        targets=None
    ):

        B,T = x.shape

        # token embeddings
        tokenEmbeddings = self.tokenEmbedding(x)

        # positional embeddings
        positions = torch.arange(
            T,
            device=x.device
        )

        positionEmbeddings = self.positionEmbedding(
            positions
        )

        x = tokenEmbeddings + positionEmbeddings

       

        if True:

            x = self.blocks(x)
            # logits
            logits = self.lm_head(x)

            # loss
            B,T,C = logits.shape

            logitsFlat = logits.view(
                B*T,
                C
            )
            loss = None
            if targets is not None:
                targetsFlat = targets.view(B*T)

                loss = F.cross_entropy(
                    logitsFlat,
                    targetsFlat
                )

            return logits, loss

       
    
    def generate(
    self,
    x,
    max_new_tokens,
    Temperature
    ):

        for _ in range(max_new_tokens):

            # keep only recent context
            x_cond = x[:, -self.block_size:]
            print(x_cond)

            logits, loss = self(
                x_cond,
                targets=None
            )

            # take last token predictions
            logits = logits[:, -1, :]

            #use temperature to control response randomness
            logits/=Temperature

            # convert to probabilities
            probabilities = F.softmax(
                logits,
                dim=-1
            )

            # sample next token
            nextToken = torch.multinomial(
                probabilities,
                num_samples=1
            )

            # append token
            x = torch.cat(
                (x, nextToken),
                dim=1
            )

        return x

In [29]:
model = MiniGPT(
    vocabSize=vocabSize,
    embedding_size=32,
    block_size=8,
    num_heads=4
)

In [30]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

In [31]:
model.parameters

<bound method Module.parameters of MiniGPT(
  (dropout): Dropout(p=0.2, inplace=False)
  (tokenEmbedding): Embedding(65, 32)
  (positionEmbedding): Embedding(8, 32)
  (lm_head): Linear(in_features=32, out_features=65, bias=True)
  (blocks): Sequential(
    (0): TransformerBlock(
      (query): Linear(in_features=32, out_features=32, bias=False)
      (key): Linear(in_features=32, out_features=32, bias=False)
      (value): Linear(in_features=32, out_features=32, bias=False)
      (outputProjection): Linear(in_features=32, out_features=32, bias=True)
      (layerNorm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
      (layerNorm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
      (ffn): Sequential(
        (0): Linear(in_features=32, out_features=128, bias=True)
        (1): ReLU()
        (2): Linear(in_features=128, out_features=32, bias=True)
      )
      (norm1): RmsNorm()
      (norm2): RmsNorm()
    )
    (1): TransformerBlock(
      (query): Linear(in_feature

In [32]:
@torch.no_grad()
def estimate_loss():

    out = {}

    model.eval()

    for split in ['train', 'test']:

        losses = torch.zeros(50)

        for k in range(50):

            x,y = get_batch(split)

            logits, loss = model(x,y)

            losses[k] = loss.item()

        out[split] = losses.mean()

    model.train()

    return out

In [33]:
for step in range(5000):

    if step % 500 == 0:

        losses = estimate_loss()

        print(
            "step:",
            step,
            "train loss:",
            losses['train'].item(),
            "test loss:",
            losses['test'].item()
        )

    x,y = get_batch('train')

    logits, loss = model(x,y)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

step: 0 train loss: 4.519796371459961 test loss: 4.501981258392334
step: 500 train loss: 2.5711560249328613 test loss: 2.6364078521728516
step: 1000 train loss: 2.445901393890381 test loss: 2.484400987625122
step: 1500 train loss: 2.4529547691345215 test loss: 2.3893392086029053
step: 2000 train loss: 2.425374984741211 test loss: 2.3160927295684814
step: 2500 train loss: 2.399432897567749 test loss: 2.260563850402832
step: 3000 train loss: 2.3532488346099854 test loss: 2.3121018409729004
step: 3500 train loss: 2.2333767414093018 test loss: 2.3326032161712646
step: 4000 train loss: 2.2246670722961426 test loss: 2.2732138633728027
step: 4500 train loss: 2.2964589595794678 test loss: 2.265597105026245


In [34]:
context = torch.zeros(
    (1,1),
    dtype=torch.long
)

In [35]:
context

tensor([[0]])

In [36]:
generated = model.generate(
    context,
    max_new_tokens=100,
    Temperature=0.7
)

tensor([[0]])
tensor([[ 0, 32]])
tensor([[ 0, 32, 46]])
tensor([[ 0, 32, 46, 47]])
tensor([[ 0, 32, 46, 47, 57]])
tensor([[ 0, 32, 46, 47, 57, 43]])
tensor([[ 0, 32, 46, 47, 57, 43,  1]])
tensor([[ 0, 32, 46, 47, 57, 43,  1, 39]])
tensor([[32, 46, 47, 57, 43,  1, 39, 52]])
tensor([[46, 47, 57, 43,  1, 39, 52, 42]])
tensor([[47, 57, 43,  1, 39, 52, 42,  1]])
tensor([[57, 43,  1, 39, 52, 42,  1, 39]])
tensor([[43,  1, 39, 52, 42,  1, 39, 52]])
tensor([[ 1, 39, 52, 42,  1, 39, 52, 42]])
tensor([[39, 52, 42,  1, 39, 52, 42,  1]])
tensor([[52, 42,  1, 39, 52, 42,  1, 39]])
tensor([[42,  1, 39, 52, 42,  1, 39, 52]])
tensor([[ 1, 39, 52, 42,  1, 39, 52, 42]])
tensor([[39, 52, 42,  1, 39, 52, 42,  1]])
tensor([[52, 42,  1, 39, 52, 42,  1, 58]])
tensor([[42,  1, 39, 52, 42,  1, 58, 53]])
tensor([[ 1, 39, 52, 42,  1, 58, 53,  1]])
tensor([[39, 52, 42,  1, 58, 53,  1, 58]])
tensor([[52, 42,  1, 58, 53,  1, 58, 46]])
tensor([[42,  1, 58, 53,  1, 58, 46, 43]])
tensor([[ 1, 58, 53,  1, 58, 46, 43,  

In [37]:
decoded = ''.join(
    itos[i]
    for i in generated[0].tolist()
)

print(decoded)


Thise and and and to the shat the is siselarst wor to thure stoungeran that my most for, thas wald t
